# Robustness: optimizer, width, teacher and initialization

205 runs, five paired seeds, 100k updates each. The main outcome is raw accumulated pairwise penalty at fixed budget; normalized, equal-activity and matched-loss measures are controls. Read `docs/robustness.md` for the full protocol.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import Image, display

root=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/"src/neural_competition").exists())
results=root/"runs/pier_study/robustness/analysis"
summary=pd.read_csv(results/"summary.csv")
effects=pd.read_csv(results/"effects.csv")
seeds=pd.read_csv(results/"per_seed_results.csv")
structure=pd.read_csv(results/"neuron_structure.csv")

def figure(name):
    display(Image(filename=str(results/f"{name}.png")))

def show_case(condition):
    figure(condition.replace("/","_")+"_trajectory")


## Main result

Width helps fitting across the core grid. A smaller pairwise penalty is robust for momentum but not Adam. The compact figure compares widest versus minimal student; ratios below one favor the wider student. Intervals bootstrap paired seeds and are exploratory.

In [ ]:
display(pd.read_csv(results/"core_widest.csv"))
figure("robustness_core_effects_reference_compact")

In [ ]:
figure("robustness_fitting")
figure("robustness_raw_penalty")

## Controls for useful activity and learning stage

Equal U is equal accumulated standalone helpful gains, not equal actual progress. The complete loss-window summary uses MSE 0.2→0.02. Many narrow Adam runs do not cover this window; missing comparisons are not zero penalty. All-width detailed matched-loss curves only include bins visited by all five seeds at every width.

In [ ]:
figure("robustness_normalized_controls")
figure("robustness_core_effects_reference")
figure("robustness_loss_coverage")

## Frequencies

The broad frequency tests J>0 among pairs of helpful standalone updates. The strict frequency requires the second helpful update to become harmful after the first. Magnitude C+ includes all unordered pairs and is a different quantity.

In [ ]:
figure("robustness_unfavorable_pair_frequency")
figure("robustness_strict_pair_frequency")

## Teacher realizations and smaller learning rate

Teacher functions are separate experimental conditions. Five initialization seeds per teacher do not supply five independent teacher draws.

In [ ]:
figure("robustness_controls_effects_reference")
display(summary[(summary.view=="reference") & (summary.role!="core")][["condition","width","window_mse","window_test_mse","penalty","S_plus"]])

## Targeted initialization follow-up

After the core sweep showed all-negative outgoing coefficients in narrow Adam models, outgoing initialization std 0.1 and 1 were tested. This is a post-hoc diagnostic. All incoming weights and minibatch sequences are paired. Nonzero outgoing initialization changes initial predictions and hidden-layer gradients, not only signs.

In [ ]:
figure("adam_initialization_control")
display(pd.read_csv(results/"initialization_summary.csv"))

If all outgoing coefficients are nonpositive, ReLU implies $f(x)\leq0$.
Then empirical ordinary MSE is at least $\mathbb E[(y_+)^2]$. This explains a
concrete limitation of the observed narrow Adam state, not its entire error
or a proof of why the optimizer arrived there.

In [ ]:
display(structure[(structure.profile=="adam_001") & (structure.teacher_width==8) & (structure.width==8)][["condition","seed","positive_outputs","positive_active","inactive_on_train","global_nonpositive","nonpositive_predictor_mse_bound","final_mse"]])

## Exact loss accounting

For each update, ΔL=H−G+C+−C−. On the fixed reference these changes telescope across training. In the Adam counterexample, larger pairwise cost is compensated by larger net standalone gains G−H. These are trajectory accounting statements, not a causal decomposition of the width advantage.

In [ ]:
display(pd.read_csv(results/"adam_reference_budget.csv"))

## Explore individual conditions

Each group has its own six-panel trajectory figure: MSE, raw unfavorable/net costs, normalized cost, equal-U cost and matched-loss cost. Edit the condition below to inspect another group.

In [ ]:
print("Available conditions:", summary.condition.unique().tolist())
show_case("k16_teacher2027/momentum_01")
show_case("k8_teacher2027/adam_001")

## Validation and raw data

In [ ]:
display(pd.read_csv(results/"audit.csv").groupby("view")[["max_local_error","telescope_error","reference_adjacency_error"]].max())
print(json.loads((results/"independent_audit.json").read_text()))

folder=results.parent/"k8_teacher2027/adam_001"
config=json.loads((folder/"config.json").read_text())
updates=np.load(folder/"updates.npy",mmap_mode="r")
print("Raw shape:",updates.shape,"; views:",config["views"])
print("Fields:",config["fields"])


## Limits

Finite 100k horizon, five seeds, four teacher realizations and one Gaussian input dataset. All target variances are normalized but their means differ. The finite-step pair penalty depends on optimizer and step strength and includes diminishing returns. This sweep does not establish a causal mechanism or test every other interference definition from the earlier cross-batch study.